<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/VQ_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lpips

In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time
import numpy as np

In [4]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [5]:
class Residual(nn.Module):
    def __init__(self, num_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(num_channels, num_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return x + self.block(x)

class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=64, num_channels=3):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.BatchNorm2d(hidden_dim // 2),
            nn.ReLU(),
            nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.BatchNorm2d(hidden_dim * 2),
            nn.ReLU(),
            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.BatchNorm2d(hidden_dim * 4),
            nn.ReLU(),
            nn.Conv2d(hidden_dim * 4, embedding_dim, kernel_size=4, stride=2, padding=1),           # 32 -> 16
        )
        self.residual = Residual(embedding_dim)

    def forward(self, x):
        x = self.conv(x)
        return self.residual(x)

class Decoder(nn.Module):
    def __init__(self, embedding_dim=64, hidden_dim=128, num_channels=3):
        super().__init__()

        self.residual = Residual(embedding_dim)
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, hidden_dim * 4, kernel_size=4, stride=2, padding=1),                      # 512 -> 256
            nn.BatchNorm2d(hidden_dim * 4),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1),             # 256 -> 128
            nn.BatchNorm2d(hidden_dim * 2),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1),              # 128 -> 64
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1),          # 64 -> 32
            nn.BatchNorm2d(hidden_dim // 2),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim // 2, num_channels, kernel_size=4, stride=2, padding=1),           # 32 -> 16
        )

    def forward(self, x):
        x = self.residual(x)
        x = self.conv(x)
        return torch.tanh(x)

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=64, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        #self.embeddings.weight.data.uniform_(-1.0/self.num_embeddings, 1.0/self.num_embeddings)
        self.embeddings.weight.data.uniform_(-1.0/self.embedding_dim, 1.0/self.embedding_dim)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        active_codes = torch.unique(encoding_indices).numel()
        #print(active_codes)

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        return quantized, loss, perplexity

class VQ_VAE(nn.Module):
    def __init__(self, num_embeddings=512, embedding_dim=64, hidden_dim=128, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss, perplexity = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss, perplexity


perceptual loss compares features maps, instead of raw pixels:

In [6]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()

        for param in self.slice.parameters():
            param.requires_grad = False

        # ImageNet normalization statistics
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def normalize(self, x):
      x = (x + 1.0) / 2.0
      return (x - self.mean) / self.std

    def forward(self, real, fake):
        real = self.slice(self.normalize(real))
        fake = self.slice(self.normalize(fake))
        return F.mse_loss(fake, real)

In [7]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VQ_VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9))
#perceptual_loss_fn = PerceptualLoss().to(device)
perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 145MB/s] 


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth


In [10]:
model.train()

epochs = 20

for epoch in range(epochs):
    epoch_start = time.time()
    total_loss, epoch_perplexity = 0.0, 0.0

    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        data_recon, vq_loss, perplexity = model(data)
        #recon_loss = F.mse_loss(data_recon, data)
        #loss = recon_loss + vq_loss
        data = F.interpolate(data, size=(256, 256), mode='bilinear', align_corners=False)
        data_recon = F.interpolate(data_recon, size=(256, 256), mode='bilinear', align_corners=False)
        percept_loss = perceptual_loss_fn(data_recon, data).mean()
        loss = percept_loss + vq_loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        epoch_perplexity += perplexity.item()

        #if batch_idx % 100 == 0:
        #    print(f'Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')

    print(f'====> Epoch: {epoch} | Average loss: {total_loss / len(train_loader):.4f} | Perplexity: {epoch_perplexity/len(train_loader):.2f}')



====> Epoch: 0 | Average loss: 3.0763 | Perplexity: 12.06
====> Epoch: 1 | Average loss: 2.6807 | Perplexity: 9.45
====> Epoch: 2 | Average loss: 2.7631 | Perplexity: 8.49
====> Epoch: 3 | Average loss: 2.3419 | Perplexity: 8.65
====> Epoch: 4 | Average loss: 1.9493 | Perplexity: 8.74
====> Epoch: 5 | Average loss: 1.7039 | Perplexity: 9.05
====> Epoch: 6 | Average loss: 1.5127 | Perplexity: 9.25
====> Epoch: 7 | Average loss: 1.4342 | Perplexity: 9.47
====> Epoch: 8 | Average loss: 1.2986 | Perplexity: 9.72
====> Epoch: 9 | Average loss: 1.2312 | Perplexity: 9.92
====> Epoch: 10 | Average loss: 1.1923 | Perplexity: 10.02
====> Epoch: 11 | Average loss: 1.1236 | Perplexity: 10.17
====> Epoch: 12 | Average loss: 1.0990 | Perplexity: 10.33
====> Epoch: 13 | Average loss: 1.0511 | Perplexity: 10.47
====> Epoch: 14 | Average loss: 1.0069 | Perplexity: 10.57
====> Epoch: 15 | Average loss: 0.9730 | Perplexity: 10.70
====> Epoch: 16 | Average loss: 0.9586 | Perplexity: 10.58
====> Epoch: 17 